# Day 21: Multimodal – CLIP & BLIP

## 1. Load CLIP model and processor

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("CLIP model loaded on", device)

## 2. Zero‑shot classification with CLIP

In [ ]:
# Sample image (cat)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png"
response = requests.get(url)
img = Image.open(BytesIO(response.content))
display(img)  # or img.show()

candidate_labels = ["a cat", "a dog", "a bird", "a car"]
inputs = processor(text=candidate_labels, images=img, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]

for label, prob in zip(candidate_labels, probs):
    print(f"{label}: {prob:.4f}")

## 3. Load BLIP for image captioning

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

inputs = blip_processor(images=img, return_tensors="pt").to(device)
out = blip_model.generate(**inputs, max_length=20)
caption = blip_processor.decode(out[0], skip_special_tokens=True)
print("BLIP caption:", caption)

## 4. Compare CLIP and BLIP
CLIP tells you *what class* the image belongs to; BLIP describes *what’s happening*.